# Tables
## Universidad ICESI 
### David Mauricio Orozco Rios
### author: Davoroz06 - IG

In [22]:
# libraries
import pandas as pd
import numpy as np
import os
from datetime import datetime
from scipy.stats import gmean
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# options for data display
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 20)

# project paths and shared helpers (see src/config.py)
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import *  # wd, wd_dp, wd_db, wd_dpr, wd_re, wd_rp, wd_rt ...
from utils import homogenize_text


In [23]:
regular_window = 7
reference_window = 7
case = "Comparison"

## Load data:

In [24]:
# load retailer data
data = pd.read_csv(wd_dpr + "Data_Regular_Price_{}_{}_{}.csv".format(case, regular_window, reference_window))

# Convert 'fecha' to datetime format
data['fecha'] = pd.to_datetime(data['fecha'])

In [25]:
date_range = pd.date_range(start=data['fecha'].min(), end=data['fecha'].max())

In [26]:
date_range

DatetimeIndex(['2024-04-19', '2024-04-20', '2024-04-21', '2024-04-22',
               '2024-04-23', '2024-04-24', '2024-04-25', '2024-04-26',
               '2024-04-27', '2024-04-28', '2024-04-29', '2024-04-30',
               '2024-05-01', '2024-05-02', '2024-05-03', '2024-05-04',
               '2024-05-05', '2024-05-06', '2024-05-07', '2024-05-08',
               '2024-05-09', '2024-05-10', '2024-05-11', '2024-05-12',
               '2024-05-13', '2024-05-14', '2024-05-15', '2024-05-16',
               '2024-05-17', '2024-05-18', '2024-05-19', '2024-05-20',
               '2024-05-21', '2024-05-22', '2024-05-23', '2024-05-24',
               '2024-05-25', '2024-05-26', '2024-05-27', '2024-05-28',
               '2024-05-29', '2024-05-30', '2024-05-31', '2024-06-01',
               '2024-06-02', '2024-06-03', '2024-06-04', '2024-06-05',
               '2024-06-06', '2024-06-07', '2024-06-08', '2024-06-09',
               '2024-06-10', '2024-06-11', '2024-06-12', '2024-06-13',
      

### Table 1. Descripive statistics

Data Base description

In [27]:
# Convertir la columna de fecha a formato de fecha y agregar una columna de mes
data['fecha'] = pd.to_datetime(data['fecha'])

# Calcular las estadísticas descriptivas por tienda
tabla_1 = data[~data["precio"].isna()].groupby(['tienda']).apply(lambda x: pd.Series({
    'Days': (x['fecha'].max() - x['fecha'].min()).days,
    'Products': x['descripcion'].nunique(),
    'Observations': len(x['precio']),
    'Price mean': x['precio'].mean(),
    'Price sd': x['precio'].std(),
    'Price median': x['precio'].median(),
    'Log Price mean': np.log(x['precio']).mean(),
    'Log Price sd': np.log(x['precio']).std(),
    'Log Price median': np.log(x['precio']).median()
    }))

# Calculate total row (ignoring groups)
total_row = pd.Series({
    'Days': (data['fecha'].max() - data['fecha'].min()).days,
    'Products': data['descripcion'].nunique(),
    'Observations': data['precio'].count(),
    'Price mean': data['precio'].mean(),
    'Price sd': data['precio'].std(),
    'Price median': data['precio'].median(),
    'Log Price mean': np.log(data['precio']).mean(),
    'Log Price sd': np.log(data['precio']).std(),
    'Log Price median': np.log(data['precio']).median()
}, name="Total")

# Append a new row to the DataFrame
tabla_1 = pd.concat([tabla_1, total_row.to_frame().T]).rename_axis("tienda")

# Format numeric columns to integers (no decimals)
tabla_1['Days'] = tabla_1['Days'].astype(int)
tabla_1['Products'] = tabla_1['Products'].astype(int)
tabla_1['Observations'] = tabla_1['Observations'].astype(int)

# Format price columns to $ with 2 decimal places
tabla_1['Price mean'] = tabla_1['Price mean'].apply(lambda x: f"${x:,.2f}")
tabla_1['Price sd'] = tabla_1['Price sd'].apply(lambda x: f"${x:,.2f}")
tabla_1['Price median'] = tabla_1['Price median'].apply(lambda x: f"${x:,.2f}")

tabla_1['Log Price mean'] = tabla_1['Log Price mean'].apply(lambda x: f"${x:,.2f}")
tabla_1['Log Price sd'] = tabla_1['Log Price sd'].apply(lambda x: f"${x:,.2f}")
tabla_1['Log Price median'] = tabla_1['Log Price median'].apply(lambda x: f"${x:,.2f}")

pd.DataFrame(tabla_1)

,Days,Products,Observations,Price mean,Price sd,Price median,Log Price mean,Log Price sd,Log Price median
tienda,,,,,,,,,
C,60,327,18705,"$7,864.93","$8,407.51","$5,490.00",$8.63,$0.80,$8.61
A1,61,3769,141570,"$95,886.25","$451,101.95","$29,700.00",$10.30,$1.38,$10.30
A2,58,3355,140108,"$113,519.55","$606,925.65","$19,600.00",$10.06,$1.49,$9.88
B,61,1339,40523,"$35,305.24","$56,921.73","$10,400.00",$9.48,$1.41,$9.25
Total,61,8761,340906,"$91,102.54","$487,083.90","$19,980.00",$10.01,$1.47,$9.90


## Table 2. Price setting agg

In [28]:
psdf = pd.read_excel(wd_re + "Price_Setting_{}_{}_{}.xlsx".format(case, regular_window, reference_window))

In [29]:
# Calcular las estadísticas descriptivas por tienda
psdf_median = psdf.groupby('tienda').apply(lambda x: pd.Series({
    'Price Change Fr': x['Price Change Fr'].median(),
    'Price Implied Duration': x['Price Implied Duration'].median(),
    'Price Change Si': x['Price Change Si'].median(),
    'Price Change Di': x['Price Change Di'].median(),
    'RPE Change Fr': x['RPE Change Fr'].median(),
    'RPE Implied Duration': x['RPE Implied Duration'].median(),
    'RPE Change Si': x['RPE Change Si'].median(),
    'RPE Change Di': x['RPE Change Di'].median(),
    'RPN Change Fr': x['RPN Change Fr'].median(),
    'RPN Implied Duration': x['RPN Implied Duration'].median(),
    'RPN Change Si': x['RPN Change Si'].median(),
    'RPN Change Di': x['RPN Change Di'].median()
}))

# Calculate total row (ignoring groups)
total_row = pd.Series({
    'Price Change Fr': psdf_median['Price Change Fr'].median(),
    'Price Implied Duration': psdf_median['Price Implied Duration'].median(),
    'Price Change Si': psdf_median['Price Change Si'].median(),
    'Price Change Di': psdf_median['Price Change Di'].median(),
    'RPE Change Fr': psdf_median['RPE Change Fr'].median(),
    'RPE Implied Duration': psdf_median['RPE Implied Duration'].median(),
    'RPE Change Si': psdf_median['RPE Change Si'].median(),
    'RPE Change Di': psdf_median['RPE Change Di'].median(),
    'RPN Change Fr': psdf_median['RPN Change Fr'].median(),
    'RPN Implied Duration': psdf_median['RPN Implied Duration'].median(),
    'RPN Change Si': psdf_median['RPN Change Si'].median(),
    'RPN Change Di': psdf_median['RPN Change Di'].median()
}, name="Total")

# Append a new row to the DataFrame
psdf_median = pd.concat([psdf_median, total_row.to_frame().T]).rename_axis("tienda")

# Format price columns to $ with 2 decimal places
psdf_median['Price Change Fr'] = psdf_median['Price Change Fr'].apply(lambda x: f"%{x*100:,.2f}")
psdf_median['Price Implied Duration'] = psdf_median['Price Implied Duration'].apply(lambda x: f"{x:,.2f}")
psdf_median['Price Change Si'] = psdf_median['Price Change Si'].apply(lambda x: f"%{x*100:,.2f}")
psdf_median['Price Change Di'] = psdf_median['Price Change Di'].apply(lambda x: f"%{x*100:,.2f}")

psdf_median['RPE Change Fr'] = psdf_median['RPE Change Fr'].apply(lambda x: f"%{x*100:,.2f}")
psdf_median['RPE Implied Duration'] = psdf_median['RPE Implied Duration'].apply(lambda x: f"{x:,.2f}")
psdf_median['RPE Change Si'] = psdf_median['RPE Change Si'].apply(lambda x: f"%{x*100:,.2f}")
psdf_median['RPE Change Di'] = psdf_median['RPE Change Di'].apply(lambda x: f"%{x*100:,.2f}")

psdf_median['RPN Change Fr'] = psdf_median['RPN Change Fr'].apply(lambda x: f"%{x*100:,.2f}")
psdf_median['RPN Implied Duration'] = psdf_median['RPN Implied Duration'].apply(lambda x: f"{x:,.2f}")
psdf_median['RPN Change Si'] = psdf_median['RPN Change Si'].apply(lambda x: f"%{x*100:,.2f}")
psdf_median['RPN Change Di'] = psdf_median['RPN Change Di'].apply(lambda x: f"%{x*100:,.2f}")

pd.DataFrame(psdf_median)

,Price Change Fr,Price Implied Duration,Price Change Si,Price Change Di,RPE Change Fr,RPE Implied Duration,RPE Change Si,RPE Change Di,RPN Change Fr,RPN Implied Duration,RPN Change Si,RPN Change Di
tienda,,,,,,,,,,,,
C,%1.69,58.50,%0.28,%100.00,%1.69,58.50,%0.28,%100.00,%1.69,58.50,%0.28,%100.00
A1,%5.00,19.50,%16.25,%50.00,%2.56,38.50,%14.88,%50.00,%2.63,37.50,%13.88,%50.00
A2,%5.41,18.00,%22.31,%50.00,%2.22,44.50,%19.48,%50.00,%4.44,22.00,%22.31,%50.00
B,%0.00,58.00,%19.44,%50.00,%0.00,58.00,%17.57,%50.00,%0.00,58.00,%19.43,%50.00
Total,%3.35,38.75,%17.85,%50.00,%1.96,51.25,%16.22,%50.00,%2.16,47.75,%16.65,%50.00


In [30]:
# Calcular las estadísticas descriptivas por tienda
psdf_mean = psdf.groupby('tienda').apply(lambda x: pd.Series({
    'Price Change Fr': x['Price Change Fr'].mean(),
    'Price Implied Duration': x['Price Implied Duration'].mean(),
    'Price Change Si': x['Price Change Si'].mean(),
    'Price Change Di': x['Price Change Di'].mean(),
    'RPE Change Fr': x['RPE Change Fr'].mean(),
    'RPE Implied Duration': x['RPE Implied Duration'].mean(),
    'RPE Change Si': x['RPE Change Si'].mean(),
    'RPE Change Di': x['RPE Change Di'].mean(),
    'RPN Change Fr': x['RPN Change Fr'].mean(),
    'RPN Implied Duration': x['RPN Implied Duration'].mean(),
    'RPN Change Si': x['RPN Change Si'].mean(),
    'RPN Change Di': x['RPN Change Di'].mean()
}))

# Calculate total row (ignoring groups)
total_row = pd.Series({
    'Price Change Fr': psdf_mean['Price Change Fr'].median(),
    'Price Implied Duration': psdf_mean['Price Implied Duration'].median(),
    'Price Change Si': psdf_mean['Price Change Si'].median(),
    'Price Change Di': psdf_mean['Price Change Di'].median(),
    'RPE Change Fr': psdf_mean['RPE Change Fr'].median(),
    'RPE Implied Duration': psdf_mean['RPE Implied Duration'].median(),
    'RPE Change Si': psdf_mean['RPE Change Si'].median(),
    'RPE Change Di': psdf_mean['RPE Change Di'].median(),
    'RPN Change Fr': psdf_mean['RPN Change Fr'].median(),
    'RPN Implied Duration': psdf_mean['RPN Implied Duration'].median(),
    'RPN Change Si': psdf_mean['RPN Change Si'].median(),
    'RPN Change Di': psdf_mean['RPN Change Di'].median()
}, name="Total")

# Append a new row to the DataFrame
psdf_mean = pd.concat([psdf_mean, total_row.to_frame().T]).rename_axis("tienda")

# Format price columns to $ with 2 decimal places
psdf_mean['Price Change Fr'] = psdf_mean['Price Change Fr'].apply(lambda x: f"%{x*100:,.2f}")
psdf_mean['Price Implied Duration'] = psdf_mean['Price Implied Duration'].apply(lambda x: f"{x:,.2f}")
psdf_mean['Price Change Si'] = psdf_mean['Price Change Si'].apply(lambda x: f"%{x*100:,.2f}")
psdf_mean['Price Change Di'] = psdf_mean['Price Change Di'].apply(lambda x: f"%{x*100:,.2f}")

psdf_mean['RPE Change Fr'] = psdf_mean['RPE Change Fr'].apply(lambda x: f"%{x*100:,.2f}")
psdf_mean['RPE Implied Duration'] = psdf_mean['RPE Implied Duration'].apply(lambda x: f"{x:,.2f}")
psdf_mean['RPE Change Si'] = psdf_mean['RPE Change Si'].apply(lambda x: f"%{x*100:,.2f}")
psdf_mean['RPE Change Di'] = psdf_mean['RPE Change Di'].apply(lambda x: f"%{x*100:,.2f}")

psdf_mean['RPN Change Fr'] = psdf_mean['RPN Change Fr'].apply(lambda x: f"%{x*100:,.2f}")
psdf_mean['RPN Implied Duration'] = psdf_mean['RPN Implied Duration'].apply(lambda x: f"{x:,.2f}")
psdf_mean['RPN Change Si'] = psdf_mean['RPN Change Si'].apply(lambda x: f"%{x*100:,.2f}")
psdf_mean['RPN Change Di'] = psdf_mean['RPN Change Di'].apply(lambda x: f"%{x*100:,.2f}")

pd.DataFrame(psdf_mean)

,Price Change Fr,Price Implied Duration,Price Change Si,Price Change Di,RPE Change Fr,RPE Implied Duration,RPE Change Si,RPE Change Di,RPN Change Fr,RPN Implied Duration,RPN Change Si,RPN Change Di
tienda,,,,,,,,,,,,
C,%0.98,57.88,%1.61,%73.89,%0.98,57.88,%1.61,%73.89,%0.98,57.88,%1.61,%73.89
A1,%9.95,30.20,%17.29,%48.17,%4.10,37.02,%16.53,%52.20,%6.96,34.94,%16.31,%49.49
A2,%10.86,26.97,%22.10,%48.23,%2.75,40.88,%20.30,%52.21,%5.51,32.84,%20.81,%48.37
B,%2.34,49.99,%24.30,%53.97,%1.18,51.62,%19.61,%57.41,%1.74,50.68,%24.24,%54.11
Total,%6.15,40.10,%19.70,%51.10,%1.96,46.25,%18.07,%54.81,%3.62,42.81,%18.56,%51.80


In [31]:
ps_table = psdf_median.merge(psdf_mean, on = "tienda", suffixes=('_median', '_mean'))

In [32]:
ps_table

,Price Change Fr_median,Price Implied Duration_median,Price Change Si_median,Price Change Di_median,RPE Change Fr_median,RPE Implied Duration_median,RPE Change Si_median,RPE Change Di_median,RPN Change Fr_median,RPN Implied Duration_median,...,Price Change Si_mean,Price Change Di_mean,RPE Change Fr_mean,RPE Implied Duration_mean,RPE Change Si_mean,RPE Change Di_mean,RPN Change Fr_mean,RPN Implied Duration_mean,RPN Change Si_mean,RPN Change Di_mean
tienda,,,,,,,,,,,,,,,,,,,,,
C,%1.69,58.50,%0.28,%100.00,%1.69,58.50,%0.28,%100.00,%1.69,58.50,...,%1.61,%73.89,%0.98,57.88,%1.61,%73.89,%0.98,57.88,%1.61,%73.89
A1,%5.00,19.50,%16.25,%50.00,%2.56,38.50,%14.88,%50.00,%2.63,37.50,...,%17.29,%48.17,%4.10,37.02,%16.53,%52.20,%6.96,34.94,%16.31,%49.49
A2,%5.41,18.00,%22.31,%50.00,%2.22,44.50,%19.48,%50.00,%4.44,22.00,...,%22.10,%48.23,%2.75,40.88,%20.30,%52.21,%5.51,32.84,%20.81,%48.37
B,%0.00,58.00,%19.44,%50.00,%0.00,58.00,%17.57,%50.00,%0.00,58.00,...,%24.30,%53.97,%1.18,51.62,%19.61,%57.41,%1.74,50.68,%24.24,%54.11
Total,%3.35,38.75,%17.85,%50.00,%1.96,51.25,%16.22,%50.00,%2.16,47.75,...,%19.70,%51.10,%1.96,46.25,%18.07,%54.81,%3.62,42.81,%18.56,%51.80


## Table 3. Transition matrix

In [33]:
tmdf = pd.read_excel(wd_re + "Transition_matrix_{}_{}_{}.xlsx".format(case, regular_window, reference_window))

In [34]:
tmdf = tmdf[tmdf["period"]==1]

In [35]:
tm_regular = tmdf.groupby("tienda").apply(lambda x: pd.Series({
    'Mean_P_11': x["P_11_regular"].mean(),
    'Mean_P_12': x["P_12_regular"].mean(),
    'Mean_P_21': x["P_21_regular"].mean(),
    'Mean_P_22': x["P_22_regular"].mean(),
    'Median_P_11': x["P_11_regular"].median(),
    'Median_P_12': x["P_12_regular"].median(),
    'Median_P_21': x["P_21_regular"].median(),
    'Median_P_22': x["P_22_regular"].median()
}))

# Calculate total row (ignoring groups)
total_row = pd.Series({
    'Mean_P_11': tmdf["P_11_regular"].mean(),
    'Mean_P_12': tmdf["P_12_regular"].mean(),
    'Mean_P_21': tmdf["P_21_regular"].mean(),
    'Mean_P_22': tmdf["P_22_regular"].mean(),
    'Median_P_11': tmdf["P_11_regular"].median(),
    'Median_P_12': tmdf["P_12_regular"].median(),
    'Median_P_21': tmdf["P_21_regular"].median(),
    'Median_P_22': tmdf["P_22_regular"].median()
}, name="Total")

# Append a new row to the DataFrame
tm_regular = pd.concat([tm_regular, total_row.to_frame().T]).rename_axis("tienda")

In [36]:
tm_reference = tmdf.groupby("tienda").apply(lambda x: pd.Series({
    'Mean_P_11': x["P_11_reference"].mean(),
    'Mean_P_12': x["P_12_reference"].mean(),
    'Mean_P_21': x["P_21_reference"].mean(),
    'Mean_P_22': x["P_22_reference"].mean(),
    'Median_P_11': x["P_11_reference"].median(),
    'Median_P_12': x["P_12_reference"].median(),
    'Median_P_21': x["P_21_reference"].median(),
    'Median_P_22': x["P_22_reference"].median()
}))

# Calculate total row (ignoring groups)
total_row = pd.Series({
    'Mean_P_11': tmdf["P_11_reference"].mean(),
    'Mean_P_12': tmdf["P_12_reference"].mean(),
    'Mean_P_21': tmdf["P_21_reference"].mean(),
    'Mean_P_22': tmdf["P_22_reference"].mean(),
    'Median_P_11': tmdf["P_11_reference"].median(),
    'Median_P_12': tmdf["P_12_reference"].median(),
    'Median_P_21': tmdf["P_21_reference"].median(),
    'Median_P_22': tmdf["P_22_reference"].median()
}, name="Total")

# Append a new row to the DataFrame
tm_reference = pd.concat([tm_reference, total_row.to_frame().T]).rename_axis("tienda")

In [37]:
tm_table = tm_regular.merge(tm_reference, on = "tienda", suffixes=('_regular', '_reference'))

In [38]:
tm_table

,Mean_P_11_regular,Mean_P_12_regular,Mean_P_21_regular,Mean_P_22_regular,Median_P_11_regular,Median_P_12_regular,Median_P_21_regular,Median_P_22_regular,Mean_P_11_reference,Mean_P_12_reference,Mean_P_21_reference,Mean_P_22_reference,Median_P_11_reference,Median_P_12_reference,Median_P_21_reference,Median_P_22_reference
tienda,,,,,,,,,,,,,,,,
C,1.000000,0.000000,NaN,NaN,1.0,0.0,NaN,NaN,0.989789,0.010211,0.901852,0.098148,0.982456,0.017544,1.000000,0.000000
A1,0.981354,0.018646,0.626466,0.373534,1.0,0.0,0.5,0.5,0.937223,0.062777,0.678568,0.321432,0.972222,0.027778,0.666667,0.333333
A2,0.968544,0.031456,0.857273,0.142727,1.0,0.0,1.0,0.0,0.932527,0.067473,0.797070,0.202930,0.968750,0.031250,0.833333,0.166667
B,0.995850,0.004150,0.764836,0.235164,1.0,0.0,1.0,0.0,0.986472,0.013528,0.773528,0.226472,1.000000,0.000000,0.833333,0.166667
Total,0.979366,0.020634,0.753888,0.246112,1.0,0.0,1.0,0.0,0.944888,0.055112,0.745485,0.254515,0.976190,0.023810,0.750000,0.250000


## Tabla 4. Estadisticas por tipo de producto

In [39]:
data_clas = pd.read_excel(wd_data + "product_classification.xlsx")

In [40]:
pscdf = psdf.merge(data_clas, left_on='descripcion', right_on='Descripcion', how='left')

In [41]:
pscdf = pscdf.drop(["Descripcion", 'Code'], axis = 1)

In [42]:
# Calcular las estadísticas descriptivas por tienda
pscdf_mean = pscdf.groupby(['tienda', 'Group']).apply(lambda x: pd.Series({
    'Products': x['descripcion'].nunique(),
    'Price Change Fr': x['Price Change Fr'].mean(),
    'Price Implied Duration': x['Price Implied Duration'].mean(),
    'Price Change Si': x['Price Change Si'].mean(),
    'Price Change Di': x['Price Change Di'].mean(),
    'RPE Change Fr': x['RPE Change Fr'].mean(),
    'RPE Implied Duration': x['RPE Implied Duration'].mean(),
    'RPE Change Si': x['RPE Change Si'].mean(),
    'RPE Change Di': x['RPE Change Di'].mean(),
    'RPN Change Fr': x['RPN Change Fr'].mean(),
    'RPN Implied Duration': x['RPN Implied Duration'].mean(),
    'RPN Change Si': x['RPN Change Si'].mean(),
    'RPN Change Di': x['RPN Change Di'].mean()
})).reset_index()

# Calculate total row (ignoring groups)
total_row = pscdf.groupby(['Group']).apply(lambda x: pd.Series({
    'tienda': 'Total',
    'Products': x['descripcion'].nunique(),
    'Price Change Fr': x['Price Change Fr'].mean(),
    'Price Implied Duration': x['Price Implied Duration'].mean(),
    'Price Change Si': x['Price Change Si'].mean(),
    'Price Change Di': x['Price Change Di'].mean(),
    'RPE Change Fr': x['RPE Change Fr'].mean(),
    'RPE Implied Duration': x['RPE Implied Duration'].mean(),
    'RPE Change Si': x['RPE Change Si'].mean(),
    'RPE Change Di': x['RPE Change Di'].mean(),
    'RPN Change Fr': x['RPN Change Fr'].mean(),
    'RPN Implied Duration': x['RPN Implied Duration'].mean(),
    'RPN Change Si': x['RPN Change Si'].mean(),
    'RPN Change Di': x['RPN Change Di'].mean()
})).reset_index()

# Append a new row to the DataFrame
pscdf_mean = pd.concat([pscdf_mean, total_row])

# Format price columns to $ with 2 decimal places
pscdf_mean['Price Change Fr'] = pscdf_mean['Price Change Fr'].apply(lambda x: f"%{x*100:,.2f}")
pscdf_mean['Price Implied Duration'] = pscdf_mean['Price Implied Duration'].apply(lambda x: f"{x:,.2f}")
pscdf_mean['Price Change Si'] = pscdf_mean['Price Change Si'].apply(lambda x: f"%{x*100:,.2f}")
pscdf_mean['Price Change Di'] = pscdf_mean['Price Change Di'].apply(lambda x: f"%{x*100:,.2f}")

pscdf_mean['RPE Change Fr'] = pscdf_mean['RPE Change Fr'].apply(lambda x: f"%{x*100:,.2f}")
pscdf_mean['RPE Implied Duration'] = pscdf_mean['RPE Implied Duration'].apply(lambda x: f"{x:,.2f}")
pscdf_mean['RPE Change Si'] = pscdf_mean['RPE Change Si'].apply(lambda x: f"%{x*100:,.2f}")
pscdf_mean['RPE Change Di'] = pscdf_mean['RPE Change Di'].apply(lambda x: f"%{x*100:,.2f}")

pscdf_mean['RPN Change Fr'] = pscdf_mean['RPN Change Fr'].apply(lambda x: f"%{x*100:,.2f}")
pscdf_mean['RPN Implied Duration'] = pscdf_mean['RPN Implied Duration'].apply(lambda x: f"{x:,.2f}")
pscdf_mean['RPN Change Si'] = pscdf_mean['RPN Change Si'].apply(lambda x: f"%{x*100:,.2f}")
pscdf_mean['RPN Change Di'] = pscdf_mean['RPN Change Di'].apply(lambda x: f"%{x*100:,.2f}")

pd.DataFrame(pscdf_mean)

,tienda,Group,Products,Price Change Fr,Price Implied Duration,Price Change Si,Price Change Di,RPE Change Fr,RPE Implied Duration,RPE Change Si,RPE Change Di,RPN Change Fr,RPN Implied Duration,RPN Change Si,RPN Change Di
0,C,"Alcoholic Beverages, Tobacco And Narcotics",31.0,%1.63,57.08,%1.96,%17.24,%1.63,57.08,%1.96,%17.24,%1.63,57.08,%1.96,%17.24
1,C,Food And Non-Alcoholic Beverages,290.0,%0.91,58.05,%1.41,%84.80,%0.91,58.05,%1.41,%84.80,%0.91,58.05,%1.41,%84.80
2,C,Health,1.0,%0.00,60.00,%nan,%nan,%0.00,60.00,%nan,%nan,%0.00,60.00,%nan,%nan
3,C,"Personal Care, Social Protection And Miscellan...",1.0,%0.00,58.00,%nan,%nan,%0.00,58.00,%nan,%nan,%0.00,58.00,%nan,%nan
4,C,"Recreation, Sport And Culture",4.0,%1.72,50.87,%7.73,%83.33,%1.72,50.87,%7.73,%83.33,%1.72,50.87,%7.73,%83.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4,Total,Health,68.0,%10.68,29.65,%17.86,%54.37,%2.36,45.02,%13.89,%58.79,%3.22,41.83,%14.44,%55.04
5,Total,Information And Communication,161.0,%14.88,29.90,%11.69,%34.79,%5.51,33.42,%12.24,%39.18,%13.88,32.50,%11.78,%33.03
6,Total,"Personal Care, Social Protection And Miscellan...",330.0,%12.29,26.53,%25.12,%45.33,%4.53,35.65,%23.36,%48.43,%7.64,30.72,%24.83,%45.89
7,Total,"Recreation, Sport And Culture",551.0,%4.24,41.91,%26.02,%48.75,%1.91,45.88,%23.08,%51.90,%2.52,44.79,%25.48,%49.86


In [43]:
# Calcular las estadísticas descriptivas por tienda
pscdf_median = pscdf.groupby(['tienda', 'Group']).apply(lambda x: pd.Series({
    'Products': x['descripcion'].nunique(),
    'Price Change Fr': x['Price Change Fr'].median(),
    'Price Implied Duration': x['Price Implied Duration'].median(),
    'Price Change Si': x['Price Change Si'].median(),
    'Price Change Di': x['Price Change Di'].median(),
    'RPE Change Fr': x['RPE Change Fr'].median(),
    'RPE Implied Duration': x['RPE Implied Duration'].median(),
    'RPE Change Si': x['RPE Change Si'].median(),
    'RPE Change Di': x['RPE Change Di'].median(),
    'RPN Change Fr': x['RPN Change Fr'].median(),
    'RPN Implied Duration': x['RPN Implied Duration'].median(),
    'RPN Change Si': x['RPN Change Si'].median(),
    'RPN Change Di': x['RPN Change Di'].median()
})).reset_index()

# Calculate total row (ignoring groups)
total_row = pscdf.groupby(['Group']).apply(lambda x: pd.Series({
    'tienda': 'Total',
    'Products': x['descripcion'].nunique(),
    'Price Change Fr': x['Price Change Fr'].median(),
    'Price Implied Duration': x['Price Implied Duration'].median(),
    'Price Change Si': x['Price Change Si'].median(),
    'Price Change Di': x['Price Change Di'].median(),
    'RPE Change Fr': x['RPE Change Fr'].median(),
    'RPE Implied Duration': x['RPE Implied Duration'].median(),
    'RPE Change Si': x['RPE Change Si'].median(),
    'RPE Change Di': x['RPE Change Di'].median(),
    'RPN Change Fr': x['RPN Change Fr'].median(),
    'RPN Implied Duration': x['RPN Implied Duration'].median(),
    'RPN Change Si': x['RPN Change Si'].median(),
    'RPN Change Di': x['RPN Change Di'].median()
})).reset_index()

# Append a new row to the DataFrame
pscdf_median = pd.concat([pscdf_median, total_row])

# Format price columns to $ with 2 decimal places
pscdf_median['Price Change Fr'] = pscdf_median['Price Change Fr'].apply(lambda x: f"%{x*100:,.2f}")
pscdf_median['Price Implied Duration'] = pscdf_median['Price Implied Duration'].apply(lambda x: f"{x:,.2f}")
pscdf_median['Price Change Si'] = pscdf_median['Price Change Si'].apply(lambda x: f"%{x*100:,.2f}")
pscdf_median['Price Change Di'] = pscdf_median['Price Change Di'].apply(lambda x: f"%{x*100:,.2f}")

pscdf_median['RPE Change Fr'] = pscdf_median['RPE Change Fr'].apply(lambda x: f"%{x*100:,.2f}")
pscdf_median['RPE Implied Duration'] = pscdf_median['RPE Implied Duration'].apply(lambda x: f"{x:,.2f}")
pscdf_median['RPE Change Si'] = pscdf_median['RPE Change Si'].apply(lambda x: f"%{x*100:,.2f}")
pscdf_median['RPE Change Di'] = pscdf_median['RPE Change Di'].apply(lambda x: f"%{x*100:,.2f}")

pscdf_median['RPN Change Fr'] = pscdf_median['RPN Change Fr'].apply(lambda x: f"%{x*100:,.2f}")
pscdf_median['RPN Implied Duration'] = pscdf_median['RPN Implied Duration'].apply(lambda x: f"{x:,.2f}")
pscdf_median['RPN Change Si'] = pscdf_median['RPN Change Si'].apply(lambda x: f"%{x*100:,.2f}")
pscdf_median['RPN Change Di'] = pscdf_median['RPN Change Di'].apply(lambda x: f"%{x*100:,.2f}")

pd.DataFrame(pscdf_median)

,tienda,Group,Products,Price Change Fr,Price Implied Duration,Price Change Si,Price Change Di,RPE Change Fr,RPE Implied Duration,RPE Change Si,RPE Change Di,RPN Change Fr,RPN Implied Duration,RPN Change Si,RPN Change Di
0,C,"Alcoholic Beverages, Tobacco And Narcotics",31.0,%1.75,56.50,%0.22,%0.00,%1.75,56.50,%0.22,%0.00,%1.75,56.50,%0.22,%0.00
1,C,Food And Non-Alcoholic Beverages,290.0,%1.69,58.50,%0.29,%100.00,%1.69,58.50,%0.29,%100.00,%1.69,58.50,%0.29,%100.00
2,C,Health,1.0,%0.00,60.00,%nan,%nan,%0.00,60.00,%nan,%nan,%0.00,60.00,%nan,%nan
3,C,"Personal Care, Social Protection And Miscellan...",1.0,%0.00,58.00,%nan,%nan,%0.00,58.00,%nan,%nan,%0.00,58.00,%nan,%nan
4,C,"Recreation, Sport And Culture",4.0,%1.72,57.50,%0.37,%100.00,%1.72,57.50,%0.37,%100.00,%1.72,57.50,%0.37,%100.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4,Total,Health,68.0,%5.00,19.50,%18.17,%50.00,%0.00,58.00,%14.40,%50.00,%0.00,58.00,%15.19,%50.00
5,Total,Information And Communication,161.0,%5.00,19.50,%6.78,%33.33,%2.86,34.50,%10.12,%36.67,%2.70,36.50,%6.33,%27.78
6,Total,"Personal Care, Social Protection And Miscellan...",330.0,%6.82,14.16,%24.90,%50.00,%2.50,39.50,%22.31,%50.00,%4.44,22.00,%24.34,%50.00
7,Total,"Recreation, Sport And Culture",551.0,%0.00,57.00,%22.36,%50.00,%0.00,58.00,%18.63,%50.00,%0.00,58.00,%22.31,%50.00


In [44]:
psc_table = pscdf_median.merge(pscdf_mean, on = ["Group" ,"tienda", "Products"], suffixes=('_median', '_mean'))

In [45]:
psc_table

,tienda,Group,Products,Price Change Fr_median,Price Implied Duration_median,Price Change Si_median,Price Change Di_median,RPE Change Fr_median,RPE Implied Duration_median,RPE Change Si_median,...,Price Change Si_mean,Price Change Di_mean,RPE Change Fr_mean,RPE Implied Duration_mean,RPE Change Si_mean,RPE Change Di_mean,RPN Change Fr_mean,RPN Implied Duration_mean,RPN Change Si_mean,RPN Change Di_mean
0,C,"Alcoholic Beverages, Tobacco And Narcotics",31.0,%1.75,56.50,%0.22,%0.00,%1.75,56.50,%0.22,...,%1.96,%17.24,%1.63,57.08,%1.96,%17.24,%1.63,57.08,%1.96,%17.24
1,C,Food And Non-Alcoholic Beverages,290.0,%1.69,58.50,%0.29,%100.00,%1.69,58.50,%0.29,...,%1.41,%84.80,%0.91,58.05,%1.41,%84.80,%0.91,58.05,%1.41,%84.80
2,C,Health,1.0,%0.00,60.00,%nan,%nan,%0.00,60.00,%nan,...,%nan,%nan,%0.00,60.00,%nan,%nan,%0.00,60.00,%nan,%nan
3,C,"Personal Care, Social Protection And Miscellan...",1.0,%0.00,58.00,%nan,%nan,%0.00,58.00,%nan,...,%nan,%nan,%0.00,58.00,%nan,%nan,%0.00,58.00,%nan,%nan
4,C,"Recreation, Sport And Culture",4.0,%1.72,57.50,%0.37,%100.00,%1.72,57.50,%0.37,...,%7.73,%83.33,%1.72,50.87,%7.73,%83.33,%1.72,50.87,%7.73,%83.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34,Total,Health,68.0,%5.00,19.50,%18.17,%50.00,%0.00,58.00,%14.40,...,%17.86,%54.37,%2.36,45.02,%13.89,%58.79,%3.22,41.83,%14.44,%55.04
35,Total,Information And Communication,161.0,%5.00,19.50,%6.78,%33.33,%2.86,34.50,%10.12,...,%11.69,%34.79,%5.51,33.42,%12.24,%39.18,%13.88,32.50,%11.78,%33.03
36,Total,"Personal Care, Social Protection And Miscellan...",330.0,%6.82,14.16,%24.90,%50.00,%2.50,39.50,%22.31,...,%25.12,%45.33,%4.53,35.65,%23.36,%48.43,%7.64,30.72,%24.83,%45.89
37,Total,"Recreation, Sport And Culture",551.0,%0.00,57.00,%22.36,%50.00,%0.00,58.00,%18.63,...,%26.02,%48.75,%1.91,45.88,%23.08,%51.90,%2.52,44.79,%25.48,%49.86


In [47]:
with pd.ExcelWriter(wd_rt + "Tables_{}_{}_{}.xlsx".format(case, regular_window, reference_window)) as writer:
    tabla_1.to_excel(writer, sheet_name='Table_1', index=True)
    ps_table.to_excel(writer, sheet_name='PS_table', index=True)
    tm_table.to_excel(writer, sheet_name='TM_table', index=True)
    psc_table.to_excel(writer, sheet_name='PSC_table', index=True)